# 🚀 Project Setup — Educational Quickstart Blueprint

**AI Learning Playground — Environment Configuration & Model Download**

Run this notebook **once** before opening any of the four starter notebooks.
It installs all dependencies, validates your GPU, authenticates with Hugging Face,
and downloads the quantized GGUF models used across all four blueprints.

---

## What This Notebook Does

| Cell | Section | Purpose |
|------|---------|---------|
| 2 | CUDA Configuration | Sets environment variables — **must run before any torch import** |
| 4 | PyTorch Installation | Installs `torch+cu121`; verifies GPU binding (2–5 min) |
| 6 | Library Imports | Imports `torch`, `os`, `sys`, `platform`; confirms core availability |
| 8 | GPU Validation | 4-step test: detection → alloc → matmul → cleanup |
| 10 | AI Library Install | Installs `transformers`, `diffusers`, `mlflow`, `streamlit`, etc. (3–7 min) |
| 12 | Infrastructure Test | Verifies all library imports; prints version numbers; pass/fail summary |
| 14 | Hugging Face Auth | Guided token entry; saves credentials for gated-model downloads |
| 16 | Model Download | Downloads all 5 GGUF models to `/home/jovyan/local/` |
| 18 | Setup Summary | GPU info, auth status, model paths, next steps |
| 19 | Quick Reference | Code snippets for Zephyr, FLUX, Whisper, XTTS |

## After Setup — Starter Notebooks

| Notebook | Capability | Model Used |
|----------|-----------|------------|
| [chatbot-starter.ipynb](chatbot-starter.ipynb) | Conversational AI | Zephyr 7B Beta Q5_K_M |
| [document-analyzer-starter.ipynb](document-analyzer-starter.ipynb) | Document Q&A (RAG) | Llama 3.1 8B Q6_K_L |
| [image-gen-starter.ipynb](image-gen-starter.ipynb) | Text-to-Image | FLUX.1-dev Q4_K_S |
| [voice-assistant-starter.ipynb](voice-assistant-starter.ipynb) | Voice AI (STT + TTS) | Whisper V3 Turbo + XTTS v2 |

 ## 1. Environment Configuration

Set **all required environment variables** before any `import torch` call.
CUDA variables are read by the runtime at import time — changing them afterwards has no effect.
The remaining variables configure HP AI Studio services so you never need to set them manually.

### CUDA Variables
| Variable | Value | Why |
|----------|-------|-----|
| `CUDA_VISIBLE_DEVICES` | `"0"` | Restricts PyTorch to GPU 0; avoids accidental multi-GPU fragmentation |
| `PYTORCH_CUDA_ALLOC_CONF` | `"expandable_segments:True"` | Reduces memory fragmentation by letting the allocator grow segments dynamically |
| `CUDA_LAUNCH_BLOCKING` | `"0"` | Async kernel launches (set to `"1"` only when debugging CUDA errors) |

### HP AI Studio Service Variables (Spec 4.3)
| Variable | Value | Purpose |
|----------|-------|---------|
| `MLFLOW_TRACKING_URI` | `http://localhost:5000` | HP AI Studio MLflow tracking server (default port) |
| `GRADIO_SERVER_NAME` | `0.0.0.0` | Network-accessible Gradio binding for AI Studio proxy |
| `GRADIO_SERVER_PORT` | `7860` | Standard Gradio port |
| `HF_HOME` | `/data/huggingface` | Centralized Hugging Face cache — survives workspace restarts |
| `MODELS_DIR` | `/data/models` | Standardized model storage path |
| `TRANSFORMERS_CACHE` | `/data/huggingface/hub` | Transformer model cache directory |

> ⚠️ **Run this cell first** — before importing `torch` or any other GPU/AI library.

In [1]:
import os
import sys
import time

# ── CUDA environment — must be set before any torch import ───────────────────

# Restrict execution to GPU device 0 (avoids accidental multi-GPU problems)
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# Allow PyTorch's memory allocator to grow segments dynamically.
# This significantly reduces OOM errors when loading large GGUF models.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Synchronous CUDA launches — useful for debugging CUDA errors (shows exact line).
# Keep at "0" for normal use; set to "1" only when chasing CUDA errors.
os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "0")

# ── HP AI Studio service environment (Spec 4.3) ───────────────────────────────

# MLflow tracking URI — HP AI Studio's built-in MLflow runs on localhost:5000
os.environ.setdefault("MLFLOW_TRACKING_URI", "http://localhost:5000")

# Gradio server binding — 0.0.0.0 makes the UI accessible through the AI Studio proxy
os.environ.setdefault("GRADIO_SERVER_NAME", "0.0.0.0")
os.environ.setdefault("GRADIO_SERVER_PORT", "7860")

# Hugging Face cache — persistent directory that survives workspace restarts
os.environ.setdefault("HF_HOME", "/data/huggingface")
os.environ.setdefault("TRANSFORMERS_CACHE", "/data/huggingface/hub")

# Standardized models directory for manual model management
os.environ.setdefault("MODELS_DIR", "/data/models")

# ── Ensure project root is on the Python path so `src.*` imports work ─────────
sys.path.insert(0, "..")

start_time = time.time()

print("─" * 60)
print("  Environment Configuration  —  Spec 4.3")
print("─" * 60)
print("  CUDA:")
print(f"    CUDA_VISIBLE_DEVICES     : {os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"    PYTORCH_CUDA_ALLOC_CONF  : {os.environ['PYTORCH_CUDA_ALLOC_CONF']}")
print(f"    CUDA_LAUNCH_BLOCKING     : {os.environ['CUDA_LAUNCH_BLOCKING']}")
print()
print("  HP AI Studio Services:")
print(f"    MLFLOW_TRACKING_URI      : {os.environ['MLFLOW_TRACKING_URI']}")
print(f"    GRADIO_SERVER_NAME       : {os.environ['GRADIO_SERVER_NAME']}")
print(f"    GRADIO_SERVER_PORT       : {os.environ['GRADIO_SERVER_PORT']}")
print()
print("  Storage Paths:")
print(f"    HF_HOME                  : {os.environ['HF_HOME']}")
print(f"    TRANSFORMERS_CACHE       : {os.environ['TRANSFORMERS_CACHE']}")
print(f"    MODELS_DIR               : {os.environ['MODELS_DIR']}")
print("─" * 60)
print("✅ Cell 1 complete — all environment variables configured (Spec 4.3)")
print("⏱️  Setup notebook started")

────────────────────────────────────────────────────────────
  Environment Configuration  —  Spec 4.3
────────────────────────────────────────────────────────────
  CUDA:
    CUDA_VISIBLE_DEVICES     : 0
    PYTORCH_CUDA_ALLOC_CONF  : expandable_segments:True
    CUDA_LAUNCH_BLOCKING     : 0

  HP AI Studio Services:
    MLFLOW_TRACKING_URI      : /phoenix/mlflow
    GRADIO_SERVER_NAME       : 0.0.0.0
    GRADIO_SERVER_PORT       : 7860

  Storage Paths:
    HF_HOME                  : /data/huggingface
    TRANSFORMERS_CACHE       : /data/huggingface/hub
    MODELS_DIR               : /data/models
────────────────────────────────────────────────────────────
✅ Cell 1 complete — all environment variables configured (Spec 4.3)
⏱️  Setup notebook started


## 2. PyTorch Installation

Install PyTorch built against **CUDA 12.1** from the official index.

> **Why a specific CUDA version?**
> PyTorch ships pre-compiled wheels for each CUDA toolkit version.
> Using an incompatible wheel (e.g., CUDA 11.x wheel on a CUDA 12.x driver) causes
> subtle failures during GPU kernel execution. The `cu121` wheel is correct for
> CUDA driver ≥ 12.1, which is standard on AI Studio environments.

> **First-run time:** 2–5 minutes (cached on subsequent runs).

In [2]:
# Install PyTorch with CUDA 12.1 support (2–5 min on first run)
%pip install torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu121

import torch

cuda_ok = torch.cuda.is_available()

print("─" * 55)
print("  PyTorch Installation")
print("─" * 55)
print(f"  PyTorch  : {torch.__version__}")
print(f"  CUDA     : {'✅ Available' if cuda_ok else '❌ Not found — GPU required'}")

if cuda_ok:
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"  GPU      : {props.name}")
    print(f"  VRAM     : {vram_gb:.1f} GB {'✅' if vram_gb >= 8 else '⚠️ (8 GB minimum recommended)'}")
    print(f"  Compute  : {props.major}.{props.minor}")
else:
    print("  ⚠️ GPU not detected. All models in this project require a CUDA-capable GPU.")
print("─" * 55)
print(f"✅ Cell 2 complete — PyTorch {torch.__version__} installed")

Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.
───────────────────────────────────────────────────────
  PyTorch Installation
───────────────────────────────────────────────────────
  PyTorch  : 2.5.1+cu121
  CUDA     : ✅ Available
  GPU      : NVIDIA RTX A4000
  VRAM     : 16.7 GB ✅
  Compute  : 8.6
───────────────────────────────────────────────────────
✅ Cell 2 complete — PyTorch 2.5.1+cu121 installed


## 3. Library Imports

Confirm that the core Python standard library and PyTorch are importable and print
the runtime environment details.
This snapshot helps reproduce bugs: paste the output when asking for support.

In [3]:
import os
import sys
import platform
import torch

print("─" * 55)
print("  System Information")
print("─" * 55)
print(f"  Python   : {sys.version.split()[0]}")
print(f"  Platform : {platform.system()} {platform.release()}")
print(f"  Arch     : {platform.machine()}")
print()
print("  Core PyTorch")
print(f"  torch    : {torch.__version__}")
print(f"  CUDA     : {torch.version.cuda or 'None'}")
print(f"  cuDNN    : {torch.backends.cudnn.version() or 'None'}")
print("─" * 55)
print("✅ Cell 3 complete — core imports confirmed")

───────────────────────────────────────────────────────
  System Information
───────────────────────────────────────────────────────
  Python   : 3.12.7
  Platform : Linux 6.8.0-90-generic
  Arch     : x86_64

  Core PyTorch
  torch    : 2.5.1+cu121
  CUDA     : 12.1
  cuDNN    : 90100
───────────────────────────────────────────────────────
✅ Cell 3 complete — core imports confirmed


## 4. GPU Validation

Run a 4-step functional test to confirm your GPU is correctly bound to PyTorch and can
perform the operations that the AI models rely on.

| Step | Test | What it checks |
|------|------|---------------|
| 1 | CUDA detection | `torch.cuda.is_available()` returns `True` |
| 2 | Memory allocation | Can allocate a 4 MB tensor on GPU memory |
| 3 | Matrix multiply | Can run a 512×512 fp32 matmul on GPU |
| 4 | VRAM cleanup | `torch.cuda.empty_cache()` + `gc.collect()` succeed |

All 4 steps must pass before proceeding to model downloads.

In [4]:
import gc
import torch

results = []

# ── Step 1: CUDA detection ────────────────────────────────────────────────────
cuda_ok = torch.cuda.is_available()
results.append(("CUDA detection", cuda_ok,
                f"Device: {torch.cuda.get_device_name(0)}" if cuda_ok else "CUDA not found"))
print(f"[{'✅ PASS' if cuda_ok else '❌ FAIL'}] Step 1 — CUDA detection")

if cuda_ok:
    # ── Step 2: GPU memory allocation ────────────────────────────────────────
    try:
        # Allocate a 1024×1024 float32 tensor (~4 MB) on the GPU
        t = torch.zeros(1024, 1024, dtype=torch.float32, device="cuda")
        size_mb = t.element_size() * t.nelement() / 1e6
        results.append(("Memory allocation", True, f"Allocated {size_mb:.1f} MB on GPU"))
        del t
        print(f"[✅ PASS] Step 2 — GPU memory allocation ({size_mb:.1f} MB)")
    except Exception as e:
        results.append(("Memory allocation", False, str(e)))
        print(f"[❌ FAIL] Step 2 — GPU memory allocation: {e}")

    # ── Step 3: Matrix multiply ───────────────────────────────────────────────
    try:
        # 512×512 matmul is a minimal smoke-test for cuBLAS
        a = torch.randn(512, 512, device="cuda")
        b = torch.randn(512, 512, device="cuda")
        c = torch.matmul(a, b)
        results.append(("Matrix multiply", True, f"512×512 matmul → result shape {tuple(c.shape)}"))
        del a, b, c
        print(f"[✅ PASS] Step 3 — Matrix multiply (512×512)")
    except Exception as e:
        results.append(("Matrix multiply", False, str(e)))
        print(f"[❌ FAIL] Step 3 — Matrix multiply: {e}")

    # ── Step 4: VRAM cleanup ──────────────────────────────────────────────────
    try:
        torch.cuda.empty_cache()   # Release PyTorch's internal VRAM cache
        gc.collect()               # Free Python-side garbage
        free_vram = torch.cuda.mem_get_info()[0] / 1e9
        results.append(("VRAM cleanup", True, f"Free VRAM after cleanup: {free_vram:.1f} GB"))
        print(f"[✅ PASS] Step 4 — VRAM cleanup ({free_vram:.1f} GB free)")
    except Exception as e:
        results.append(("VRAM cleanup", False, str(e)))
        print(f"[❌ FAIL] Step 4 — VRAM cleanup: {e}")

# ── Summary ───────────────────────────────────────────────────────────────────
passed = sum(1 for _, ok, _ in results if ok)
total  = len(results)
print()
print("─" * 55)
print(f"  GPU Validation: {passed}/{total} steps passed")
print("─" * 55)
for name, ok, msg in results:
    print(f"  {'✅' if ok else '❌'} {name}: {msg}")
print("─" * 55)

if passed == total:
    print("✅ Cell 4 complete — GPU is fully functional")
else:
    print("⚠️  Fix the failing steps before downloading models.")

[✅ PASS] Step 1 — CUDA detection
[✅ PASS] Step 2 — GPU memory allocation (4.2 MB)
[✅ PASS] Step 3 — Matrix multiply (512×512)
[✅ PASS] Step 4 — VRAM cleanup (8.2 GB free)

───────────────────────────────────────────────────────
  GPU Validation: 4/4 steps passed
───────────────────────────────────────────────────────
  ✅ CUDA detection: Device: NVIDIA RTX A4000
  ✅ Memory allocation: Allocated 4.2 MB on GPU
  ✅ Matrix multiply: 512×512 matmul → result shape (512, 512)
  ✅ VRAM cleanup: Free VRAM after cleanup: 8.2 GB
───────────────────────────────────────────────────────
✅ Cell 4 complete — GPU is fully functional


## 5. System Dependencies (Spec 4.2)

Install the system-level packages required by the AI libraries in this blueprint.
These are **OS packages**, not Python packages — they must be installed via `apt-get`.

| Package | Purpose | Required by |
|---------|---------|-------------|
| `ffmpeg` | Audio/video transcoding — decodes MP3, OGG, FLAC → WAV | Whisper (pywhispercpp), XTTS v2 (CoquiTTS) |
| `portaudio19-dev` | System audio driver for real-time capture/playback | `pyaudio` (Python binding for PortAudio) |
| `git-lfs` | Large file storage for model downloads from Hugging Face | Manual `git clone` of model repos |

> **Why `git-lfs` if we use `hf_hub_download()`?**
> This project uses `hf_hub_download()` (HTTP-based, no git required), so `git-lfs` isn't
> actively called. However, students exploring the ecosystem may use `git clone` directly, and
> the spec mandates pre-installation for all nine analyzed projects.

> **First-run time:** ~30 seconds (cached after first install).

In [5]:
import subprocess

print("─" * 60)
print("  System Dependencies Install  —  Spec 4.2")
print("─" * 60)

# ── apt-get packages ──────────────────────────────────────────────────────────
# Suppress progress output but capture errors
install_cmd = [
    "sudo", "apt-get", "install", "-y", "-qq",
    "ffmpeg",           # Audio/video transcoding for Whisper + TTS
    "portaudio19-dev",  # PortAudio system driver for pyaudio
    "git-lfs",          # Large file support for HF model repos
]

try:
    subprocess.run(
        ["sudo", "apt-get", "update", "-qq"],
        check=True, capture_output=True
    )
    result = subprocess.run(install_cmd, check=True, capture_output=True, text=True)
    print("  ✅ apt-get packages installed: ffmpeg, portaudio19-dev, git-lfs")
except subprocess.CalledProcessError as e:
    print(f"  ⚠️  apt-get failed: {e.stderr[:200]}")
    print("  (This is expected if packages are already installed or apt is unavailable)")

# ── Configure git-lfs ─────────────────────────────────────────────────────────
try:
    subprocess.run(
        ["git", "lfs", "install", "--skip-smudge"],
        check=True, capture_output=True
    )
    print("  ✅ git-lfs configured (--skip-smudge: does not auto-download LFS files on clone)")
except subprocess.CalledProcessError:
    print("  ⚠️  git-lfs install failed (install manually if needed)")

# ── Verification ──────────────────────────────────────────────────────────────
print()
print("  Verification:")
checks = [
    ("ffmpeg",   ["ffmpeg", "-version"]),
    ("git-lfs",  ["git", "lfs", "version"]),
]
for name, cmd in checks:
    try:
        out = subprocess.run(cmd, capture_output=True, text=True)
        first_line = out.stdout.strip().split("\n")[0]
        print(f"    ✅ {name:<12} {first_line}")
    except Exception as e:
        print(f"    ❌ {name:<12} not found: {e}")

# portaudio19-dev is a C library — verify via dpkg
try:
    out = subprocess.run(
        ["dpkg", "-s", "portaudio19-dev"],
        capture_output=True, text=True
    )
    version_line = next(
        (l for l in out.stdout.split("\n") if l.startswith("Version")), ""
    )
    print(f"    ✅ portaudio19-dev  {version_line}")
except Exception:
    print("    ⚠️  portaudio19-dev  status unknown (dpkg not available)")

print()
print("─" * 60)
print("✅ Cell 5 complete — system dependencies installed")
print("   ffmpeg: Whisper + TTS audio transcoding")
print("   portaudio19-dev: required by pyaudio Python package")
print("   git-lfs: HF large-file model support")

────────────────────────────────────────────────────────────
  System Dependencies Install  —  Spec 4.2
────────────────────────────────────────────────────────────
  ✅ apt-get packages installed: ffmpeg, portaudio19-dev, git-lfs
  ✅ git-lfs configured (--skip-smudge: does not auto-download LFS files on clone)

  Verification:
    ✅ ffmpeg       ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
    ✅ git-lfs      git-lfs/3.0.2 (GitHub; linux amd64; go 1.18.1)
    ✅ portaudio19-dev  Version: 19.6.0-1.1

────────────────────────────────────────────────────────────
✅ Cell 5 complete — system dependencies installed
   ffmpeg: Whisper + TTS audio transcoding
   portaudio19-dev: required by pyaudio Python package
   git-lfs: HF large-file model support


## 5. AI Library Install

Install all project AI dependencies from `requirements.txt`.

This installs: `transformers`, `diffusers` (≥ 0.31 for FLUX GGUF support),
`accelerate`, `mlflow`, `streamlit`, `langchain`, `chromadb`, `pywhispercpp`
(for GGUF Whisper STT), `TTS` (Coqui XTTS v2), `soundfile`, `datasets`, and more.

> **First-run time:** 3–7 minutes (cached on subsequent runs).

In [6]:
# Install all project dependencies (3–7 min on first run, fast when cached)
%pip install --quiet -r ../requirements.txt

print("✅ Cell 5 complete — all AI libraries installed")

ERROR: Ignored the following versions that require a different python version: 0.0.10.2 Requires-Python >=3.6.0, <3.9; 0.0.10.3 Requires-Python >=3.6.0, <3.9; 0.0.11 Requires-Python >=3.6.0, <3.9; 0.0.12 Requires-Python >=3.6.0, <3.9; 0.0.13.1 Requires-Python >=3.6.0, <3.9; 0.0.13.2 Requires-Python >=3.6.0, <3.9; 0.0.14.1 Requires-Python >=3.6.0, <3.9; 0.0.15 Requires-Python >=3.6.0, <3.9; 0.0.15.1 Requires-Python >=3.6.0, <3.9; 0.0.9 Requires-Python >=3.6.0, <3.9; 0.0.9.1 Requires-Python >=3.6.0, <3.9; 0.0.9.2 Requires-Python >=3.6.0, <3.9; 0.0.9a10 Requires-Python >=3.6.0, <3.9; 0.0.9a9 Requires-Python >=3.6.0, <3.9; 0.1.0 Requires-Python >=3.6.0, <3.10; 0.1.1 Requires-Python >=3.6.0, <3.10; 0.1.2 Requires-Python >=3.6.0, <3.10; 0.1.3 Requires-Python >=3.6.0, <3.10; 0.10.0 Requires-Python >=3.7.0, <3.11; 0.10.1 Requires-Python >=3.7.0, <3.11; 0.10.2 Requires-Python >=3.7.0, <3.11; 0.11.0 Requires-Python >=3.7.0, <3.11; 0.11.1 Requires-Python >=3.7.0, <3.11; 0.12.0 Requires-Python >=3

## 6. Infrastructure Test

Import every critical AI library and confirm its version number.
A 5/5 pass means the environment is fully wired and all starter notebooks can run.

In [7]:
sections = {
    "4.2.1 — Core ML/AI Frameworks": [
        ("transformers",    "import transformers",    "transformers.__version__"),
        ("diffusers",       "import diffusers",       "diffusers.__version__"),
        ("accelerate",      "import accelerate",      "accelerate.__version__"),
        ("bitsandbytes",    "import bitsandbytes",    "bitsandbytes.__version__"),
        ("vllm",            "import vllm",            "vllm.__version__"),
        ("xformers",        "import xformers",        "xformers.__version__"),
        ("safetensors",     "import safetensors",     "safetensors.__version__"),
        ("sentencepiece",   "import sentencepiece",   "sentencepiece.__version__"),
    ],
    "4.2.2 — Interface & Deployment": [
        ("mlflow",          "import mlflow",          "mlflow.__version__"),
        ("huggingface_hub", "import huggingface_hub", "huggingface_hub.__version__"),
        ("fastapi",         "import fastapi",         "fastapi.__version__"),
        ("pydantic",        "import pydantic",        "pydantic.__version__"),
        ("datasets",        "import datasets",        "datasets.__version__"),
    ],
    "4.2.3 — Document & Media Processing": [
        ("pypdf",              "import pypdf",                                        "pypdf.__version__"),
        ("pymupdf (fitz)",     "import fitz",                                         "fitz.__version__"),
        ("invisible_watermark","from imwatermark import WatermarkEncoder as _WM",     "\"(installed)\""),
        ("PIL (pillow)",       "from PIL import Image as _P",                         "\"(installed)\""),
        ("cv2 (opencv)",       "import cv2",                                           "cv2.__version__"),
    ],
    "4.2.4 — Audio & Voice Processing": [
        ("pywhispercpp",   "from pywhispercpp.model import Model as _W",   "\"(installed)\""),
        ("TTS (CoquiTTS)", "from TTS.api import TTS as _T",                "\"(installed)\""),
        ("pyaudio",        "import pyaudio",                               "\"(installed)\""),
        ("torchaudio",     "import torchaudio",                            "torchaudio.__version__"),
        ("librosa",        "import librosa",                               "librosa.__version__"),
        ("soundfile",      "import soundfile",                             "soundfile.__version__"),
        ("sounddevice",    "import sounddevice",                           "sounddevice.__version__"),
    ],
    "4.2.5 — Agentic AI & RAG": [
        ("langchain",            "import langchain",            "langchain.__version__"),
        ("chromadb",             "import chromadb",             "chromadb.__version__"),
        ("faiss-gpu (faiss)",    "import faiss",                "\"(installed)\""),
        ("sentence_transformers","import sentence_transformers","sentence_transformers.__version__"),
    ],
    "4.2.6 — Monitoring & Utilities": [
        ("psutil",       "import psutil",                                  "psutil.__version__"),
        ("gputil",       "import GPUtil",                                  "\"(installed)\""),
        ("plotly",       "import plotly",                                  "plotly.__version__"),
        ("pyyaml",       "import yaml",                                    "yaml.__version__"),
        ("python-dotenv","from dotenv import load_dotenv as _ld",          "\"(installed)\""),
    ],
}

total_passed = 0
total_count  = 0
section_results = []

print("═" * 60)
print("  Infrastructure Test — AI Libraries (Spec 4.2)")
print("═" * 60)

for section_name, libs in sections.items():
    print(f"\n  {section_name}")
    print("  " + "─" * 56)
    sec_passed = 0
    for name, stmt, ver_expr in libs:
        try:
            exec(stmt, {})
            version = eval(ver_expr)
            print(f"  ✅ {name:<24} {version}")
            sec_passed += 1
        except Exception as e:
            print(f"  ❌ {name:<24} {str(e)[:50]}")
        total_count += 1
    total_passed += sec_passed
    section_results.append((section_name, sec_passed, len(libs)))
    print(f"  → {sec_passed}/{len(libs)} passed")

print("\n" + "═" * 60)
print(f"  Grand Total: {total_passed}/{total_count} libraries available")
print("═" * 60)

if total_passed == total_count:
    print("✅ Cell 6 complete — all libraries verified")
else:
    failed = total_count - total_passed
    print(f"⚠️  {failed} library/libraries missing.")
    print("   Re-run the AI Library Install cell and then retry this cell.")


════════════════════════════════════════════════════════════
  Infrastructure Test — AI Libraries (Spec 4.2)
════════════════════════════════════════════════════════════

  4.2.1 — Core ML/AI Frameworks
  ────────────────────────────────────────────────────────


/opt/conda/lib/python3.12/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


  ❌ transformers             name 'transformers' is not defined
  ❌ diffusers                name 'diffusers' is not defined
  ❌ accelerate               name 'accelerate' is not defined


WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.5.1+cu121)
    Python  3.10.19 (you have 3.12.7)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


  ❌ bitsandbytes             name 'bitsandbytes' is not defined
  ❌ vllm                     No module named 'vllm'
  ❌ xformers                 name 'xformers' is not defined
  ❌ safetensors              name 'safetensors' is not defined
  ❌ sentencepiece            name 'sentencepiece' is not defined
  → 0/8 passed

  4.2.2 — Interface & Deployment
  ────────────────────────────────────────────────────────
  ❌ mlflow                   name 'mlflow' is not defined
  ❌ huggingface_hub          name 'huggingface_hub' is not defined
  ❌ fastapi                  name 'fastapi' is not defined
  ❌ pydantic                 name 'pydantic' is not defined
  ❌ datasets                 name 'datasets' is not defined
  → 0/5 passed

  4.2.3 — Document & Media Processing
  ────────────────────────────────────────────────────────
  ❌ pypdf                    name 'pypdf' is not defined
  ❌ pymupdf (fitz)           name 'fitz' is not defined
  ❌ invisible_watermark      No module named 'imwatermark'

## 7. Hugging Face Authentication

Some models in this project require a **free Hugging Face account** and an **access token**.

### Steps to get your token:
1. Create a free account at [huggingface.co](https://huggingface.co/join)
2. Go to **Settings → Access Tokens** → [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Click **"New token"** → Name it (e.g., `ai-studio`) → select **"Read"** → **"Generate token"**
4. Copy the token — it starts with `hf_...`

### For FLUX.1-dev (gated model):
FLUX.1-dev requires accepting the license before download:
5. Visit [https://huggingface.co/black-forest-labs/FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)
6. Click **"Access repository"** and accept the license agreement
7. Wait a few minutes for your access to be approved (usually instant)

> **Security note:** Your token is entered via `getpass` — it will not appear on screen
> and will not be saved to the notebook file.

In [ ]:
import getpass
from huggingface_hub import login, whoami

hf_token = getpass.getpass("🔑 Paste your Hugging Face token (hidden): ")

try:
    # login() caches the token in ~/.cache/huggingface/token
    # add_to_git_credential=False keeps it out of git credential storage
    login(token=hf_token, add_to_git_credential=False)
    user = whoami()
    print(f"✅ Authenticated as : {user['name']}")
    print(f"   Account type     : {user.get('type', 'user')}")
    print(f"   Email            : {user.get('email', '(not set)')}")
    hf_auth_ok = True
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("   Check your token at https://huggingface.co/settings/tokens")
    hf_auth_ok = False

print()
print("✅ Cell 7 complete" if hf_auth_ok else "⚠️  Fix auth before model downloads")

## 8. Model Download

Download all quantized GGUF models used by the four starter notebooks.
Each download is **skipped automatically** if the file already exists — re-running this
cell is safe and will only fetch what is missing.

> ⚠️ **Total download size:** ~40 GB. Ensure you have sufficient disk space at
> `/home/jovyan/local/` before proceeding.

> **Note on FLUX.1-dev:** This model is gated. You must have accepted the license
> at [https://huggingface.co/black-forest-labs/FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)
> and completed Cell 7 (HF Auth) before this download will succeed.

| # | Capability | Model | Repo | Size |
|---|-----------|-------|------|------|
| 1 | Chatbot | Zephyr 7B Beta Q5_K_M | `TheBloke/zephyr-7B-beta-GGUF` | ~4.8 GB |
| 2 | Document + Voice LLM | Llama 3.1 8B Q6_K_L | `bartowski/Meta-Llama-3.1-8B-Instruct-GGUF` | ~6.6 GB |
| 3a | Image Gen (GGUF transformer) | FLUX.1-dev Q4_K_S | `city96/FLUX.1-dev-gguf` | ~6.9 GB |
| 3b | Image Gen (pipeline components) | FLUX.1-dev encoders + VAE | `black-forest-labs/FLUX.1-dev` | ~22 GB |
| 4 | Voice STT | Whisper Large V3 Turbo Q4_1 | `xkeyC/whisper-large-v3-turbo-gguf` | ~0.5 GB |
| 5 | Voice TTS | XTTS v2 F16 | `GenMedLabs/xtts-gguf` | ~2.8 GB |

In [ ]:
import os
import time
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

# ── Base directory for all locally stored models ──────────────────────────────
# This project stores models at /home/jovyan/local/ (not datafabric) so that
# students can download and own their model files directly in the JupyterHub home.
LOCAL_BASE = Path("/home/jovyan/local")
LOCAL_BASE.mkdir(parents=True, exist_ok=True)

download_results = []


def _skip_or_download(label: str, check_path: Path, download_fn) -> bool:
    """Download a model asset unless it already exists at check_path."""
    if check_path.exists():
        size_gb = (
            sum(f.stat().st_size for f in check_path.rglob("*") if f.is_file()) / 1e9
            if check_path.is_dir()
            else check_path.stat().st_size / 1e9
        )
        print(f"  ⏭️  {label}")
        print(f"       → already at {check_path} ({size_gb:.2f} GB)")
        return True
    print(f"  ⬇️  {label}")
    print(f"       → downloading to {check_path} ...")
    t0 = time.time()
    try:
        download_fn()
        elapsed = time.time() - t0
        size_gb = (
            sum(f.stat().st_size for f in check_path.rglob("*") if f.is_file()) / 1e9
            if check_path.is_dir()
            else check_path.stat().st_size / 1e9
        )
        print(f"       ✅ done in {elapsed:.0f}s ({size_gb:.2f} GB)")
        return True
    except Exception as e:
        print(f"       ❌ FAILED: {e}")
        return False


print("─" * 60)
print("  Model Download  —  /home/jovyan/local/")
print("─" * 60)

# ── 1. Chatbot LLM: Zephyr 7B Beta Q5_K_M GGUF ───────────────────────────────
#    Zephyr is a fine-tuned version of Mistral 7B optimised for instruction following.
#    Q5_K_M = 5-bit K-quantization (medium variant) — excellent quality/size trade-off.
dest_dir  = LOCAL_BASE / "zephyr-7b-beta"
dest_file = dest_dir / "zephyr-7b-beta.Q5_K_M.gguf"
dest_dir.mkdir(parents=True, exist_ok=True)
ok = _skip_or_download(
    "Chatbot LLM  — Zephyr 7B Beta Q5_K_M  (~4.8 GB)",
    dest_file,
    lambda: hf_hub_download(
        repo_id="TheBloke/zephyr-7B-beta-GGUF",
        filename="zephyr-7b-beta.Q5_K_M.gguf",
        local_dir=str(dest_dir),
    ),
)
download_results.append(("Chatbot LLM  (Zephyr 7B Beta Q5_K_M)", ok, str(dest_file)))

# ── 2. Document + Voice LLM: Meta-Llama 3.1 8B Q6_K_L GGUF ──────────────────
#    Shared between the document analyzer and voice assistant to save disk space.
#    Q6_K_L = 6-bit K-quantization (large variant) — near full-precision quality.
dest_dir  = LOCAL_BASE / "meta-llama3.1-8b-Q6"
dest_file = dest_dir / "Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf"
dest_dir.mkdir(parents=True, exist_ok=True)
ok = _skip_or_download(
    "Document + Voice LLM  — Llama 3.1 8B Q6_K_L  (~6.6 GB)",
    dest_file,
    lambda: hf_hub_download(
        repo_id="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
        filename="Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf",
        local_dir=str(dest_dir),
    ),
)
download_results.append(("Document + Voice LLM  (Llama 3.1 8B Q6_K_L)", ok, str(dest_file)))

# ── 3a. Image Gen: FLUX.1-dev GGUF transformer ────────────────────────────────
#    Q4_K_S = 4-bit K-quantization (small) — best size for GPU inference.
#    This file replaces only the transformer block in the FLUX.1-dev pipeline.
dest_dir  = LOCAL_BASE / "flux1-dev"
gguf_file = dest_dir / "flux1-dev-Q4_K_S.gguf"
dest_dir.mkdir(parents=True, exist_ok=True)
ok_transformer = _skip_or_download(
    "Image Gen GGUF transformer  — FLUX.1-dev Q4_K_S  (~6.9 GB)",
    gguf_file,
    lambda: hf_hub_download(
        repo_id="city96/FLUX.1-dev-gguf",
        filename="flux1-dev-Q4_K_S.gguf",
        local_dir=str(dest_dir),
    ),
)

# ── 3b. Image Gen: FLUX.1-dev pipeline components ─────────────────────────────
#    Downloads text encoders (T5 + CLIP), VAE, scheduler, and tokenizers.
#    The transformer weights are excluded (we use the GGUF file from step 3a).
#    ⚠️  REQUIRES: HF auth + accepted license at black-forest-labs/FLUX.1-dev
config_marker = dest_dir / "model_index.json"
ok_pipeline = _skip_or_download(
    "Image Gen pipeline components  — FLUX.1-dev encoders + VAE  (~22 GB)",
    config_marker,
    lambda: snapshot_download(
        repo_id="black-forest-labs/FLUX.1-dev",
        local_dir=str(dest_dir),
        # Exclude the large non-GGUF transformer — we use the GGUF file above instead
        ignore_patterns=["transformer/*", "*.bin"],
    ),
)
ok = ok_transformer and ok_pipeline
download_results.append(("Image Gen  (FLUX.1-dev GGUF Q4_K_S)", ok, str(dest_dir)))

# ── 4. Voice STT: Whisper Large V3 Turbo GGUF ─────────────────────────────────
#    Whisper Large V3 Turbo is a streamlined Whisper variant — faster than V3
#    with minimal quality loss. Loaded via pywhispercpp (whisper.cpp Python binding).
#    Q4_1 = 4-bit quantization, approximately 0.5 GB.
dest_dir  = LOCAL_BASE / "whisper-large-v3-turbo"
dest_file = dest_dir / "model_q4_1.gguf"
dest_dir.mkdir(parents=True, exist_ok=True)
ok = _skip_or_download(
    "Voice STT  — Whisper Large V3 Turbo Q4_1  (~0.5 GB)",
    dest_file,
    lambda: hf_hub_download(
        repo_id="xkeyC/whisper-large-v3-turbo-gguf",
        filename="model_q4_1.gguf",
        local_dir=str(dest_dir),
    ),
)
download_results.append(("Voice STT  (Whisper V3 Turbo Q4_1 GGUF)", ok, str(dest_file)))

# ── 5. Voice TTS: XTTS v2 F16 GGUF ───────────────────────────────────────────
#    XTTS v2 is Coqui's multilingual zero-shot text-to-speech model.
#    F16 = float16 full-precision weights in GGUF container format (~2.8 GB).
#    Loaded via the TTS (CoquiTTS) Python library.
dest_dir  = LOCAL_BASE / "xtts-v2"
dest_file = dest_dir / "gguf" / "xtts_v2_f16.gguf"   # mirrors the repo path
dest_dir.mkdir(parents=True, exist_ok=True)
ok = _skip_or_download(
    "Voice TTS  — XTTS v2 F16 GGUF  (~2.8 GB)",
    dest_file,
    lambda: hf_hub_download(
        repo_id="GenMedLabs/xtts-gguf",
        filename="gguf/xtts_v2_f16.gguf",
        local_dir=str(dest_dir),
    ),
)
download_results.append(("Voice TTS  (XTTS v2 F16 GGUF)", ok, str(dest_file)))

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("─" * 60)
print("  Download Summary")
print("─" * 60)
all_ok = True
for label, status, path in download_results:
    icon = "✅" if status else "❌"
    print(f"  {icon} {label}")
    print(f"       {path}")
    if not status:
        all_ok = False
print("─" * 60)
if all_ok:
    print("✅ Cell 8 complete — all models ready")
else:
    print("⚠️  Some downloads failed. Check errors above and re-run.")

## 9. Setup Summary

Full status report of the environment after completing all cells above.

In [ ]:
import gc
import torch
from pathlib import Path

LOCAL_BASE = Path("/home/jovyan/local")

# ── GPU ───────────────────────────────────────────────────────────────────────
print("─" * 60)
print("  GPU Status")
print("─" * 60)
if torch.cuda.is_available():
    props  = torch.cuda.get_device_properties(0)
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"  GPU      : {props.name}")
    print(f"  VRAM     : {props.total_memory / 1e9:.1f} GB total  |  {free_b / 1e9:.1f} GB free")
    print(f"  Compute  : {props.major}.{props.minor}")
    print(f"  Driver   : CUDA {torch.version.cuda}")
else:
    print("  ❌ GPU not available")

# ── HF Auth ───────────────────────────────────────────────────────────────────
print()
print("─" * 60)
print("  Hugging Face Auth")
print("─" * 60)
try:
    from huggingface_hub import whoami
    user = whoami()
    print(f"  ✅ Logged in as: {user['name']}")
except Exception:
    print("  ❌ Not authenticated — re-run Cell 7")

# ── Model Files ───────────────────────────────────────────────────────────────
print()
print("─" * 60)
print("  Model File Status")
print("─" * 60)
_model_checks = [
    ("Chatbot LLM  Zephyr 7B Beta Q5_K_M",
     LOCAL_BASE / "zephyr-7b-beta" / "zephyr-7b-beta.Q5_K_M.gguf"),
    ("Document LLM  Llama 3.1 8B Q6_K_L",
     LOCAL_BASE / "meta-llama3.1-8b-Q6" / "Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf"),
    ("Voice LLM  Llama 3.1 8B Q6_K_L  (shared)",
     LOCAL_BASE / "meta-llama3.1-8b-Q6" / "Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf"),
    ("Image Gen  FLUX.1-dev GGUF transformer",
     LOCAL_BASE / "flux1-dev" / "flux1-dev-Q4_K_S.gguf"),
    ("Image Gen  FLUX.1-dev pipeline",
     LOCAL_BASE / "flux1-dev" / "model_index.json"),
    ("Voice STT  Whisper V3 Turbo Q4_1",
     LOCAL_BASE / "whisper-large-v3-turbo" / "model_q4_1.gguf"),
    ("Voice TTS  XTTS v2 F16",
     LOCAL_BASE / "xtts-v2" / "gguf" / "xtts_v2_f16.gguf"),
]
all_present = True
for label, path in _model_checks:
    exists = path.exists()
    icon   = "✅" if exists else "❌"
    size   = f"({path.stat().st_size / 1e9:.2f} GB)" if exists and path.is_file() else ""
    print(f"  {icon} {label} {size}")
    print(f"       {path}")
    if not exists:
        all_present = False

# ── Next Steps ────────────────────────────────────────────────────────────────
elapsed = time.time() - start_time
print()
print("─" * 60)
print(f"  Total setup time: {elapsed:.0f}s  ({elapsed / 60:.1f} min)")
print("─" * 60)
print()
if all_present:
    print("🎉 Setup complete! Open a starter notebook to begin:")
    print()
    print("  📂 chatbot-starter.ipynb          ← Conversational AI  (Zephyr 7B)")
    print("  📂 document-analyzer-starter.ipynb ← Document Q&A  (Llama 3.1 8B)")
    print("  📂 image-gen-starter.ipynb         ← Text-to-Image  (FLUX.1-dev)")
    print("  📂 voice-assistant-starter.ipynb   ← Voice AI  (Whisper + XTTS v2)")
else:
    print("⚠️  Some models are still missing — re-run Cell 8 to retry downloads.")

## 10. Quick Reference

Common code snippets for working with the downloaded models in the starter notebooks.

### Chatbot — Zephyr 7B Beta (LlamaCpp)
```python
from langchain_community.llms import LlamaCpp

llm = LlamaCpp(
    model_path="/home/jovyan/local/zephyr-7b-beta/zephyr-7b-beta.Q5_K_M.gguf",
    n_gpu_layers=-1, n_ctx=4096, temperature=0.7,
)

# Zephyr prompt template (ChatML-like)
prompt = """<|system|>
You are a helpful assistant.</s>
<|user|>
What is the Eiffel Tower?</s>
<|assistant|>
"""
response = llm(prompt)
```

### Document Analyzer + Voice LLM — Llama 3.1 8B (LlamaCpp)
```python
from langchain_community.llms import LlamaCpp

llm = LlamaCpp(
    model_path="/home/jovyan/local/meta-llama3.1-8b-Q6/Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf",
    n_gpu_layers=-1, n_ctx=8192, temperature=0.0,
)

# Llama 3.1 prompt template
prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>
Summarise this text.<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""
```

### Image Generation — FLUX.1-dev GGUF (diffusers)
```python
import torch
from diffusers import FluxPipeline, FluxTransformer2DModel
from diffusers.utils import GGUFQuantizationConfig

MODEL_DIR = "/home/jovyan/local/flux1-dev"
transformer = FluxTransformer2DModel.from_single_file(
    f"{MODEL_DIR}/flux1-dev-Q4_K_S.gguf",
    quantization_config=GGUFQuantizationConfig(compute_dtype=torch.bfloat16),
    torch_dtype=torch.bfloat16,
)
pipe = FluxPipeline.from_pretrained(MODEL_DIR, transformer=transformer, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
image = pipe("A red cat on a spaceship", num_inference_steps=28, guidance_scale=3.5).images[0]
```

### Voice STT — Whisper V3 Turbo GGUF (pywhispercpp)
```python
from pywhispercpp.model import Model as WhisperCppModel

stt = WhisperCppModel("/home/jovyan/local/whisper-large-v3-turbo/model_q4_1.gguf")
segments = stt.transcribe("/path/to/audio.wav")
transcription = " ".join(seg.text.strip() for seg in segments)
```

### Voice TTS — XTTS v2 (TTS / CoquiTTS)
```python
from TTS.api import TTS

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")
tts.tts_to_file(text="Hello from AI Studio!", speaker_wav="ref.wav",
                language="en", file_path="output.wav")
```

### Common Error Fixes
| Error | Fix |
|-------|-----|
| `CUDA out of memory` | `torch.cuda.empty_cache(); import gc; gc.collect()` |
| `Model not found` | Re-run Cell 8 (Model Download) |
| `401 Unauthorized` | Re-run Cell 7 (HF Auth) |
| `FLUX access denied` | Accept license at huggingface.co/black-forest-labs/FLUX.1-dev |
| `Import error` | Re-run Cell 5 (AI Library Install) then kernel restart |